# SMRD stage 2 — propagation trees (BiGCN) and text + graph fusion (Colab)

Two arms, same splits and rules as stage 1 (model selection on validation macro-F1,
test evaluated once, full and dedup-clean test reported):

* **graph** — BiGCN on the propagation tree alone (comparable with published BiGCN).
* **fusion** — RoBERTa source-tweet embedding + BiGCN embedding → linear, trained end to end.

Tree caches are in the repo (`data/graphs/`), so no archives are needed here.
Checkpoints go to Drive after every epoch; re-running a cell resumes, finished seeds are skipped.

**Before switching or disconnecting the runtime, run the last cell** (flushes Drive).

In [ ]:
#@title 1. GPU check and dependencies
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('Runtime -> Change runtime type -> GPU (T4 or L4)')
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')
%pip install -q 'transformers==5.22.1' 'tokenizers==0.23.3' 'safetensors==0.8.0'     'huggingface_hub==2.4.0' 'pandas==3.1.0' 'scikit-learn==1.9.1' 'pytest==9.2.0'

In [ ]:
#@title 2. Mount Drive, get the stage2-propagation branch
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'stage2-propagation'
CKPT_ROOT = '/content/drive/MyDrive/smrd/checkpoints/stage2'
STAGE1_ROOT = '/content/drive/MyDrive/smrd/checkpoints/encoder'   # stage 1 summaries live here
os.makedirs(CKPT_ROOT, exist_ok=True)

PROJECT_ROOT = pathlib.Path('/content/smrd-reproduction')
if not PROJECT_ROOT.exists():
    !git clone -b {BRANCH} https://github.com/Miyonk/smrd-reproduction.git {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
os.chdir(PROJECT_ROOT)
!git log --oneline -1

In [ ]:
#@title 2b. Unit tests (CPU, tiny models, ~2 min)
!python -m pytest tests/test_propagation.py tests/test_encoder_pipeline.py -q

In [ ]:
#@title 3. Duplicate clusters (needed for the dedup-clean test subset)
!python -m src.run_encoder dedup --all-datasets > /dev/null && echo ok

In [ ]:
#@title 4. Tree stats + structure-only shortcut check (size/shape features, no text)
!python -m src.run_propagation stats --all-datasets

## Graph arm (BiGCN alone)

Batch 64, lr 5e-4, weight decay 1e-4, ≤ 100 epochs, patience 10 — BiGCN's settings.
Small model; each dataset should take a few minutes.

In [ ]:
#@title 5. BiGCN, all datasets, 3 seeds
for ds in ('twitter16', 'twitter15', 'pheme'):
    !python -m src.run_propagation train --dataset {ds} --arm graph --seeds 42 43 44 --ckpt-root {CKPT_ROOT}

## Fusion arm (RoBERTa + BiGCN)

Batch 16, lr 2e-5 (text) / 5e-4 (graph), linear warmup/decay, ≤ 8 epochs, patience 3.
Roughly stage 1's time per epoch.

In [ ]:
#@title 6. Fusion, Twitter16, 3 seeds
!python -m src.run_propagation train --dataset twitter16 --arm fusion --seeds 42 43 44 --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 7. Fusion, Twitter15, 3 seeds
!python -m src.run_propagation train --dataset twitter15 --arm fusion --seeds 42 43 44 --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 8. Fusion, PHEME, 3 seeds
!python -m src.run_propagation train --dataset pheme --arm fusion --seeds 42 43 44 --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 9. Summary: structure-only vs stage 1 text vs BiGCN vs fusion (copied to Drive)
import json, glob, shutil, os
def row(name, s):
    f, c = s['test_full'], s.get('test_clean') or {}
    acc = f['accuracy'] if isinstance(f['accuracy'], dict) else {'mean': f['accuracy'], 'std': 0.0}
    f1 = f['macro_f1'] if isinstance(f['macro_f1'], dict) else {'mean': f['macro_f1']}
    cacc = c.get('accuracy'); cacc = cacc['mean'] if isinstance(cacc, dict) else cacc
    print(f"{name:<44} acc {acc['mean']:.3f}±{acc['std']:.3f}  macroF1 {f1['mean']:.3f}  clean acc {cacc:.3f} (n={c.get('n')})")
for p in sorted(glob.glob('results/stage2/*.json')):
    shutil.copy(p, CKPT_ROOT)
for ds in ('twitter16', 'twitter15', 'pheme'):
    print(f'--- {ds}')
    for p in sorted(glob.glob(f'results/stage2/{ds}_*_structure_shortcut.json')):
        row('structure only (logistic regression)', json.load(open(p)))
    for p in sorted(glob.glob(f'{STAGE1_ROOT}/{ds}_*_roberta-base_summary.json')):
        row('stage 1 text (RoBERTa)', json.load(open(p)))
    for p in sorted(glob.glob(f'{CKPT_ROOT}/{ds}_*_summary.json')):
        s = json.load(open(p)); row(s['run'], s)

In [ ]:
#@title 10. Flush Drive (run before switching/disconnecting the runtime)
drive.flush_and_unmount()
print('Drive flushed and unmounted')